# Container Smoke Test

Quick validation that the container environment is working correctly.
This notebook requires **no external credentials or data downloads**.

What it checks:
1. Python version and key scientific package imports
2. NetCDF read/write round-trip
3. Cartopy map rendering
4. CTSM/CIME availability and NEON site discovery
5. NEON case creation (setup only, no build or data download)

If every cell runs without errors, the container is healthy.

## 1. Python and Package Versions

In [ ]:
import sys
print(f"Python {sys.version}")
print(f"Platform: {sys.platform}")
print()

packages = [
    "numpy", "scipy", "pandas", "xarray", "netCDF4",
    "matplotlib", "cartopy", "bokeh", "holoviews", "panel",
    "jupyterlab", "dask", "boto3", "esmpy",
]

for name in packages:
    mod = __import__(name)
    ver = getattr(mod, "__version__", "ok")
    print(f"  {name:15s} {ver}")

print("\nAll imports OK.")

## 2. NetCDF Read/Write Round-Trip

In [ ]:
import numpy as np
import xarray as xr
import tempfile, pathlib

# Create a small synthetic dataset
lats = np.linspace(-90, 90, 19)
lons = np.linspace(0, 360, 36, endpoint=False)
np.random.seed(42)
data = 260 + 30 * np.random.rand(19, 36).astype(np.float32)

ds = xr.Dataset(
    {"temperature": (["lat", "lon"], data)},
    coords={"lat": lats, "lon": lons},
    attrs={"title": "Smoke test dataset"},
)

# Write to NetCDF and read back
with tempfile.TemporaryDirectory() as tmp:
    path = pathlib.Path(tmp) / "test.nc"
    ds.to_netcdf(path)
    ds_read = xr.open_dataset(path)
    assert "temperature" in ds_read
    np.testing.assert_array_almost_equal(
        ds_read["temperature"].values, data, decimal=5
    )
    ds_read.close()

print(f"NetCDF round-trip OK ({path.stat().st_size:,} bytes written).")
ds

## 3. Cartopy Map Rendering

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig, axes = plt.subplots(
    1, 2, figsize=(14, 4),
    subplot_kw={"projection": ccrs.Robinson()},
)

# Left: coastlines and borders
ax = axes[0]
ax.set_global()
ax.add_feature(cfeature.LAND, facecolor="#e8e8e8")
ax.add_feature(cfeature.OCEAN, facecolor="#d0e4f0")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle="--")
ax.set_title("Coastlines + Borders")

# Right: pcolormesh of the synthetic temperature data
ax = axes[1]
ax.set_global()
lon2d, lat2d = np.meshgrid(lons, lats)
im = ax.pcolormesh(
    lon2d, lat2d, data,
    transform=ccrs.PlateCarree(),
    cmap="RdYlBu_r", vmin=260, vmax=290,
)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.set_title("Synthetic Temperature Field")
plt.colorbar(im, ax=ax, orientation="horizontal", pad=0.05, label="K")

plt.tight_layout()
plt.show()
print("Cartopy rendering OK.")

## 4. CTSM / CIME Availability

In [ ]:
import os
import subprocess

ctsm_root = os.environ.get("CESMROOT", "/opt/ncar/ctsm")
print(f"CTSM root:    {ctsm_root}")
print(f"CIME machine: {os.environ.get('CIME_MACHINE', 'not set')}")
print()

# Verify key scripts exist
for script in ["create_newcase", "query_config"]:
    path = os.path.join(ctsm_root, "cime", "scripts", script)
    exists = os.path.exists(path)
    print(f"  {script:20s} {'OK' if exists else 'MISSING'}")

# Verify CTSM Python modules
from ctsm import add_cime_to_path
from ctsm.path_utils import path_to_ctsm_root
print(f"\n  ctsm.path_to_ctsm_root() = {path_to_ctsm_root()}")
print("\nCTSM/CIME availability OK.")

## 5. NEON Site Discovery

This verifies that the NEON tower site configurations (usermods) are
present in the CTSM source tree. These are the per-site settings that
tell CLM how to run at each NEON tower location.

In [ ]:
import glob

neon_dir = os.path.join(ctsm_root, "cime_config", "usermods_dirs", "clm", "NEON")
sites = sorted([
    os.path.basename(d)
    for d in glob.glob(os.path.join(neon_dir, "[!d]*"))
    if os.path.isdir(d)
])

print(f"NEON usermods directory: {neon_dir}")
print(f"Sites found: {len(sites)}")
print()

# Display in columns
cols = 8
for i in range(0, len(sites), cols):
    print("  ".join(f"{s:6s}" for s in sites[i:i+cols]))

assert len(sites) >= 40, f"Expected 40+ NEON sites, found {len(sites)}"
print(f"\nNEON site discovery OK ({len(sites)} sites).")

## 6. NEON Case Creation (Setup Only)

Creates a CTSM case for the KONZ (Konza Prairie) NEON site using
`run_neon_v2 --setup-only`. This tests the full CIME case creation
workflow without downloading input data or compiling Fortran.

**Note:** This cell creates files under `/tmp/smoke_test_case/` which
are cleaned up automatically.

In [ ]:
%%bash
set -e

SITE="KONZ"
OUTPUT_ROOT="/tmp/smoke_test_neon"
rm -rf "$OUTPUT_ROOT"
mkdir -p /home/user/inputdata /home/user/scratch

echo "Creating NEON case for site: $SITE"
echo "Output root: $OUTPUT_ROOT"
echo

# Create case (setup only, no build or run)
$CESMROOT/cime/scripts/create_newcase \
    --case "$OUTPUT_ROOT/$SITE" \
    --compset I1PtClm60Bgc \
    --res CLM_USRDAT \
    --machine container \
    --run-unsupported \
    --user-mods-dirs "$CESMROOT/cime_config/usermods_dirs/clm/NEON/$SITE" \
    2>&1 | tail -5

echo
echo "Running case.setup..."
cd "$OUTPUT_ROOT/$SITE" && ./case.setup 2>&1 | tail -3

echo
echo "Case directory contents:"
ls "$OUTPUT_ROOT/$SITE/" | head -10

echo
echo "NEON case creation OK."

# Clean up
rm -rf "$OUTPUT_ROOT"

## Summary

If all cells above completed without errors, the container is healthy:

- Python scientific stack is installed and importable
- NetCDF I/O works (HDF5 + NetCDF-C libraries linked correctly)
- Cartopy can render maps (PROJ database found, shapefiles downloadable)
- CTSM and CIME are accessible
- NEON tower site configurations are present (48 sites)
- CIME case creation workflow functions end-to-end

For a full Fortran build test, run the automated test suite:
```bash
./tests/run_container_tests.sh tier2
```